# 6.16 — Layer, Group & Instance Normalization

Layer, group, and instance normalization all do the same arithmetic — subtract a mean, divide by a standard deviation, then optionally rescale and shift — but they choose different axes for the mean and variance. In this lesson, you will build the normalization rule from scratch in NumPy, inspect the exact axes used by each variant, and see why those axis choices change training stability, style behavior, and batch independence.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build normalization one idea at a time. Run each cell in order and read the printed intermediate values — every statistic is computed from scratch so the axis choices never feel like magic. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, axis reductions, broadcasting, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic activations.

▶ What you'll see: the only two libraries used in the lesson are loaded, and the random seed is fixed.

### 1. The shared normalization rule

Every variant starts with the same local calculation: choose some axes, compute their mean and variance, and convert each activation into a standardized value. The formula is

$$\hat x=\frac{x-\mu_{axes}}{\sqrt{\sigma^2_{axes}+\epsilon}}.$$

The subtraction centers the chosen group at zero, while division by the standard deviation makes one unit mean "one typical fluctuation" along those axes.

In [ ]:
x_w = np.array([2.85, 1.15, 0.40, 1.60])  # one activation vector whose scale has drifted.
mu_w = np.mean(x_w)  # mean over the chosen axis: all four features here.
var_w = np.mean((x_w - mu_w) ** 2)  # population variance over the same features.
xhat_w = (x_w - mu_w) / np.sqrt(var_w + 1e-5)  # standardized activations.

print("mean:", round(mu_w, 3), "variance:", round(var_w, 3))
print("normalized:", np.round(xhat_w, 3))

assert round(float(np.mean(xhat_w)), 6) == 0.0
assert round(float(np.var(xhat_w)), 3) == 1.0

▶ What you'll see: the raw vector becomes a new vector with mean 0 and variance about 1.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(x_w)) - 0.18, x_w, width=0.36, label="raw", color="gray")
plt.bar(np.arange(len(x_w)) + 0.18, xhat_w, width=0.36, label="normalized", color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("1: raw activations become standardized activations")
plt.xlabel("feature index")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the normalized bars are centered around zero even though the raw bars were all positive.

*Why it's done this way:* subtracting the mean removes the local offset, and dividing by the standard deviation removes the local scale. The tiny $\epsilon$ is not a modeling trick; it is a numerical guard so a nearly constant group does not divide by zero. The reason every variant can share this formula is that the axis choice, not the algebra, defines the behavior.

### 2. Layer normalization: normalize features inside each example

Layer normalization treats each example independently and normalizes across its features. For a 2-D batch shaped `(batch, features)`, each row gets its own mean and variance, so the result does not depend on which other examples happen to share the minibatch.

In [ ]:
X_w = np.array([[1.0, 2.0, 3.0, 4.0],
                [10.0, 12.0, 14.0, 16.0]])  # two examples with very different offsets and scales.
mu_ln_w = np.mean(X_w, axis=1, keepdims=True)  # one mean per row.
var_ln_w = np.var(X_w, axis=1, keepdims=True)  # one variance per row.
Y_ln_w = (X_w - mu_ln_w) / np.sqrt(var_ln_w + 1e-5)  # layer-normalized rows.

print("row means before:", mu_ln_w.ravel())
print("row means after:", np.round(np.mean(Y_ln_w, axis=1), 6))
print("row variances after:", np.round(np.var(Y_ln_w, axis=1), 3))

assert np.allclose(np.mean(Y_ln_w, axis=1), 0.0, atol=1e-6)
assert np.allclose(np.var(Y_ln_w, axis=1), 1.0, atol=1e-5)

▶ What you'll see: both rows end with mean 0 and variance 1, even though the second row started much larger.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(X_w[0], marker="o", label="raw example 0", color="gray")
plt.plot(X_w[1], marker="o", label="raw example 1", color="lightgray")
plt.plot(Y_ln_w[0], marker="s", label="LN example 0", color="teal")
plt.plot(Y_ln_w[1], marker="s", label="LN example 1", color="orange")
plt.title("2: layer norm standardizes each row separately")
plt.xlabel("feature")
plt.ylabel("activation")
plt.legend()
plt.show()

▶ What you'll see: the two normalized rows overlap in scale even though the raw rows lived on different ranges.

*Why it's done this way:* a layer receives one example's feature vector and passes it to the next operation. Normalizing across that vector keeps the feature pattern but removes its row-level offset and scale. Because the statistics are per example, layer norm works naturally with tiny batches, variable sequence lengths, and autoregressive inference where batch statistics would be awkward.

### 3. Instance normalization: normalize each channel within each example

Instance normalization is common for image-like tensors shaped `(N, C, H, W)`. For each image and each channel, it normalizes over spatial positions `(H, W)`. That means channel contrast is standardized inside each individual image, which is why instance norm often appears in style-transfer and image-generation settings.

In [ ]:
img_w = np.array([[[[1.0, 2.0], [3.0, 4.0]],
                   [[10.0, 10.0], [14.0, 14.0]]]])  # shape (1 image, 2 channels, 2x2 pixels).
mu_in_w = np.mean(img_w, axis=(2, 3), keepdims=True)  # one mean per image-channel instance.
var_in_w = np.var(img_w, axis=(2, 3), keepdims=True)  # one variance per image-channel instance.
Y_in_w = (img_w - mu_in_w) / np.sqrt(var_in_w + 1e-5)

print("instance means:", mu_in_w.reshape(-1))
print("output channel means:", np.round(np.mean(Y_in_w, axis=(2, 3)).reshape(-1), 6))
print("output channel vars:", np.round(np.var(Y_in_w, axis=(2, 3)).reshape(-1), 3))

assert np.allclose(np.mean(Y_in_w, axis=(2, 3)), 0.0, atol=1e-6)
assert np.allclose(np.var(Y_in_w, axis=(2, 3)), 1.0, atol=1e-5)

▶ What you'll see: each channel of the one image is independently centered and scaled across its four pixels.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 2.8))
ax[0].imshow(img_w[0, 0], cmap="viridis"); ax[0].set_title("raw channel 0")
ax[1].imshow(Y_in_w[0, 0], cmap="coolwarm"); ax[1].set_title("instance-norm channel 0")
for a_w in ax:
    a_w.set_xticks([]); a_w.set_yticks([])
plt.suptitle("3: spatial contrast is standardized per image-channel")
plt.show()

▶ What you'll see: the normalized channel keeps the spatial ordering but changes the values to relative contrast.

*Why it's done this way:* an image channel often has an overall brightness or contrast that is less important than the pattern within that image. Instance norm removes those per-image-channel statistics, so the remaining values describe which pixels are high or low relative to that channel's own local style.

### 4. Group normalization: split channels into groups

Group normalization sits between layer norm and instance norm. For an image tensor, it divides channels into groups and normalizes each group over channels and spatial positions. With one group it behaves like layer-style normalization over all channels and pixels of an example; with one channel per group it becomes instance normalization.

In [ ]:
G_w = 2  # split 4 channels into 2 groups of 2 channels each.
A_w = np.arange(1, 17, dtype=float).reshape(1, 4, 2, 2)  # shape (N=1, C=4, H=2, W=2).
N_w, C_w, H_w, W_w = A_w.shape
A_grouped_w = A_w.reshape(N_w, G_w, C_w // G_w, H_w, W_w)  # expose the group axis.
mu_gn_w = np.mean(A_grouped_w, axis=(2, 3, 4), keepdims=True)  # one mean per example-group.
var_gn_w = np.var(A_grouped_w, axis=(2, 3, 4), keepdims=True)  # one variance per example-group.
Y_gn_w = ((A_grouped_w - mu_gn_w) / np.sqrt(var_gn_w + 1e-5)).reshape(A_w.shape)

print("group means before:", mu_gn_w.reshape(-1))
print("group means after:", np.round(np.mean(Y_gn_w.reshape(N_w, G_w, C_w // G_w, H_w, W_w), axis=(2, 3, 4)).reshape(-1), 6))

assert np.allclose(np.mean(Y_gn_w.reshape(N_w, G_w, C_w // G_w, H_w, W_w), axis=(2, 3, 4)), 0.0, atol=1e-6)

▶ What you'll see: channels 0-1 share one statistic, and channels 2-3 share another.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["group 0 raw μ", "group 1 raw μ"], mu_gn_w.reshape(-1), color=["steelblue", "darkorange"])
plt.title("4: group norm computes one statistic per channel group")
plt.ylabel("mean before normalization")
plt.show()

▶ What you'll see: the two groups have different raw means, so each group needs its own centering.

*Why it's done this way:* channels often form related feature families. Group norm lets a small family share statistics without forcing the whole layer to share one statistic and without isolating every channel. That is the bias-variance tradeoff of the axis choice: larger groups give more samples for stable estimates, smaller groups preserve more channel-specific contrast.

### 5. Learnable gamma and beta restore useful scale

Normalization removes mean and variance on purpose, but a network may still need a feature to be large, small, shifted, or suppressed. Learnable affine parameters restore that freedom after stabilization:

$$y=\gamma\hat x+\beta.$$

The model gets stable normalized coordinates first, then learns the scale and offset that the task actually needs.

In [ ]:
hat_w = np.array([-1.342, -0.447, 0.447, 1.342])  # normalized feature values from a four-feature vector.
gamma_w = np.array([1.0, 0.5, 2.0, 0.0])  # learned feature scales.
beta_w = np.array([0.0, 1.0, -1.0, 0.25])  # learned feature shifts.
y_affine_w = gamma_w * hat_w + beta_w  # affine restoration after normalization.

print("normalized:", hat_w)
print("after gamma,beta:", np.round(y_affine_w, 3))

assert np.allclose(np.round(y_affine_w, 3), [-1.342, 0.776, -0.106, 0.25])

▶ What you'll see: one feature is damped, one is amplified, one is shifted, and one is forced to a constant beta.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(4) - 0.2, hat_w, width=0.4, label="x_hat", color="gray")
plt.bar(np.arange(4) + 0.2, y_affine_w, width=0.4, label="gamma*x_hat+beta", color="seagreen")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("5: affine parameters return task-specific scale")
plt.xlabel("feature")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: normalization does not permanently erase scale; the learned affine step can put useful scale back.

*Why it's done this way:* if normalization only forced every feature to zero mean and unit variance forever, it would restrict the model too much. The affine parameters separate two jobs: statistics stabilize optimization, while $\gamma$ and $\beta$ let the network choose the representation scale needed for the loss.

### 6. Epsilon, batch independence, and the training consequence

The normalization formula is local, but its consequences are global. Epsilon prevents numerical explosions when variance is tiny, and non-batch variants avoid coupling one example's output to other examples in the same minibatch.

In [ ]:
flat_w = np.array([3.0, 3.0, 3.0, 3.001])  # almost constant activations.
var_flat_w = np.var(flat_w)
without_eps_w = (flat_w - np.mean(flat_w)) / np.sqrt(var_flat_w)  # mathematically defined but large.
with_eps_w = (flat_w - np.mean(flat_w)) / np.sqrt(var_flat_w + 1e-3)  # damped by epsilon.

print("tiny variance:", round(float(var_flat_w), 8))
print("without eps:", np.round(without_eps_w, 3))
print("with eps:", np.round(with_eps_w, 3))

assert np.max(np.abs(with_eps_w)) < np.max(np.abs(without_eps_w))

▶ What you'll see: the same tiny differences become much less extreme when epsilon is added.

In [ ]:
batch_a_w = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 12.0, 14.0, 16.0]])
batch_b_w = np.array([[1.0, 2.0, 3.0, 4.0], [-100.0, 0.0, 100.0, 200.0]])
def ln_w(Z_w):
    return (Z_w - np.mean(Z_w, axis=1, keepdims=True)) / np.sqrt(np.var(Z_w, axis=1, keepdims=True) + 1e-5)
first_a_w = ln_w(batch_a_w)[0]
first_b_w = ln_w(batch_b_w)[0]

print("first example in batch A:", np.round(first_a_w, 3))
print("first example in batch B:", np.round(first_b_w, 3))

assert np.allclose(first_a_w, first_b_w)

▶ What you'll see: the first example's layer-normalized output is identical even when the second minibatch example changes completely.

*Why it's done this way:* stable training is not only about one value being centered. It is also about predictable gradients and reproducible behavior when batch composition changes. Layer, group, and instance normalization compute statistics inside each example, so they avoid the minibatch-dependence that can make very small-batch training noisy.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Each tiny example isolates one
> normalization mechanic from the walkthrough, prints the intermediate values, draws one
> picture, and ends with an `assert` that pins the calculation.

### ✍️ Toy 1 · Standardize one chosen group

The shared rule subtracts the chosen mean, divides by the chosen standard deviation, and produces values with mean 0 and variance 1.

In [ ]:
import numpy as np                              # arrays and axis reductions for this toy.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_x = np.array([1., 2., 3., 4., 5., 6.])       # six activations in one normalization group.
t1_mu = np.mean(t1_x)                           # group mean                         # -> 3.5
t1_var = np.var(t1_x)                           # group variance                     # -> 2.9167
t1_centered = t1_x - t1_mu                      # subtract the offset                # -> [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
t1_xhat = t1_centered / np.sqrt(t1_var + 1e-5)  # divide by standard deviation       # -> [-1.464, -0.878, -0.293, 0.293, 0.878, 1.464]

print("x:", t1_x.tolist())                     # -> [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
print("mean:", round(float(t1_mu), 3))          # -> 3.5
print("variance:", round(float(t1_var), 3))     # -> 2.917
print("centered:", t1_centered.tolist())       # -> [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
print("normalized:", np.round(t1_xhat, 3).tolist())  # -> [-1.464, -0.878, -0.293, 0.293, 0.878, 1.464]
print("mean after:", round(float(np.mean(t1_xhat)), 6))  # -> 0.0
print("variance after:", round(float(np.var(t1_xhat)), 3))  # -> 1.0

assert abs(float(np.mean(t1_xhat))) < 1e-12
assert abs(float(np.var(t1_xhat)) - 1.0) < 1e-5

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t1_x.size) - 0.18, t1_x, width=0.36, label="raw", color="gray")
plt.bar(np.arange(t1_x.size) + 0.18, t1_xhat, width=0.36, label="normalized", color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("feature")
plt.ylabel("value")
plt.title("Toy 1 · one group becomes standardized")
plt.legend()
plt.show()

▶ What you'll see: the raw increasing vector becomes centered bars whose average is 0 and whose variance is 1.

### ✍️ Toy 2 · Layer norm uses one row at a time

Layer normalization computes separate statistics for each example, so each row is centered and scaled independently.

In [ ]:
import numpy as np                              # arrays and row reductions for this toy.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_X = np.array([[1., 2., 3., 4.],              # first example with a small offset.
                 [10., 14., 18., 22.]])        # second example with a larger scale.
t2_mu = np.mean(t2_X, axis=1, keepdims=True)    # one mean per row                    # -> [[2.5], [16.0]]
t2_var = np.var(t2_X, axis=1, keepdims=True)    # one variance per row                # -> [[1.25], [20.0]]
t2_Y = (t2_X - t2_mu) / np.sqrt(t2_var + 1e-5)  # normalize each row separately.
t2_row_means = np.mean(t2_Y, axis=1)            # check row means after normalization # -> [0.0, 0.0]
t2_row_vars = np.var(t2_Y, axis=1)              # check row variances after           # -> [1.0, 1.0]

print("X:", t2_X.tolist())                     # -> [[1.0, 2.0, 3.0, 4.0], [10.0, 14.0, 18.0, 22.0]]
print("row means:", t2_mu.ravel().tolist())     # -> [2.5, 16.0]
print("row variances:", t2_var.ravel().tolist())  # -> [1.25, 20.0]
print("layer-norm rows:", np.round(t2_Y, 3).tolist())  # -> [[-1.342, -0.447, 0.447, 1.342], [-1.342, -0.447, 0.447, 1.342]]
print("row means after:", np.round(t2_row_means, 6).tolist())  # -> [0.0, 0.0]
print("row variances after:", np.round(t2_row_vars, 3).tolist())  # -> [1.0, 1.0]

assert np.allclose(t2_row_means, 0.0, atol=1e-6)
assert np.allclose(t2_row_vars, 1.0, atol=1e-5)

plt.figure(figsize=(4.8, 2.8))
plt.plot(t2_X[0], marker="o", color="gray", label="raw row 0")
plt.plot(t2_X[1], marker="o", color="lightgray", label="raw row 1")
plt.plot(t2_Y[0], marker="s", color="teal", label="LN row 0")
plt.plot(t2_Y[1], marker="s", color="orange", label="LN row 1")
plt.xlabel("feature")
plt.ylabel("activation")
plt.title("Toy 2 · layer norm standardizes rows")
plt.legend()
plt.show()

▶ What you'll see: both normalized rows land on the same small pattern despite very different raw scales.

### ✍️ Toy 3 · Instance norm uses spatial pixels per channel

Instance normalization keeps image and channel separate, then normalizes over the spatial positions inside that image-channel slice.

In [ ]:
import numpy as np                              # arrays and spatial reductions for this toy.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_img = np.array([[[[1., 2., 3.],              # channel 0, row 0.
                    [4., 5., 6.]],             # channel 0, row 1.
                   [[2., 4., 6.],              # channel 1, row 0.
                    [8., 10., 12.]]]])         # channel 1, row 1.
t3_mu = np.mean(t3_img, axis=(2, 3), keepdims=True)  # one mean per image-channel # -> [3.5, 7.0]
t3_var = np.var(t3_img, axis=(2, 3), keepdims=True)  # one variance per image-channel # -> [2.917, 11.667]
t3_Y = (t3_img - t3_mu) / np.sqrt(t3_var + 1e-5)     # normalize over H and W.
t3_means_after = np.mean(t3_Y, axis=(2, 3))           # output spatial means       # -> [[0.0, 0.0]]
t3_vars_after = np.var(t3_Y, axis=(2, 3))             # output spatial variances   # -> [[1.0, 1.0]]

print("image shape:", t3_img.shape)                  # -> (1, 2, 2, 3)
print("instance means:", np.round(t3_mu.reshape(-1), 3).tolist())  # -> [3.5, 7.0]
print("instance variances:", np.round(t3_var.reshape(-1), 3).tolist())  # -> [2.917, 11.667]
print("channel means after:", np.round(t3_means_after.reshape(-1), 6).tolist())  # -> [0.0, 0.0]
print("channel variances after:", np.round(t3_vars_after.reshape(-1), 3).tolist())  # -> [1.0, 1.0]

assert np.allclose(t3_means_after, 0.0, atol=1e-6)
assert np.allclose(t3_vars_after, 1.0, atol=1e-5)

fig, t3_ax = plt.subplots(1, 2, figsize=(5.2, 2.6))
t3_ax[0].imshow(t3_img[0, 0], cmap="viridis")
t3_ax[0].set_title("raw ch0")
t3_ax[1].imshow(t3_Y[0, 0], cmap="coolwarm")
t3_ax[1].set_title("instance ch0")
for t3_a in t3_ax:
    t3_a.set_xticks([])
    t3_a.set_yticks([])
plt.suptitle("Toy 3 · spatial slice is standardized")
plt.show()

▶ What you'll see: each channel's six pixels are recentered and rescaled without mixing channels.

### ✍️ Toy 4 · Group norm reshapes channels into groups

Group normalization exposes a group axis, computes statistics inside each group, then reshapes back to the original tensor.

In [ ]:
import numpy as np                              # arrays, reshaping, and grouped reductions for this toy.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_A = np.arange(1, 9, dtype=float).reshape(1, 4, 1, 2)  # one image, four channels, two pixels.
t4_groups = 2                                   # split four channels into two groups.
t4_N, t4_C, t4_H, t4_W = t4_A.shape             # tensor dimensions                  # -> (1, 4, 1, 2)
t4_grouped = t4_A.reshape(t4_N, t4_groups, t4_C // t4_groups, t4_H, t4_W)  # expose group axis.
t4_mu = np.mean(t4_grouped, axis=(2, 3, 4), keepdims=True)  # group means       # -> [2.5, 6.5]
t4_var = np.var(t4_grouped, axis=(2, 3, 4), keepdims=True)  # group variances   # -> [1.25, 1.25]
t4_Y = ((t4_grouped - t4_mu) / np.sqrt(t4_var + 1e-5)).reshape(t4_A.shape)  # group-normalized tensor.
t4_check = t4_Y.reshape(t4_N, t4_groups, t4_C // t4_groups, t4_H, t4_W)     # regroup for checks.
t4_means_after = np.mean(t4_check, axis=(2, 3, 4))                          # -> [[0.0, 0.0]]
t4_vars_after = np.var(t4_check, axis=(2, 3, 4))                            # -> [[1.0, 1.0]]

print("A shape:", t4_A.shape)                 # -> (1, 4, 1, 2)
print("grouped shape:", t4_grouped.shape)     # -> (1, 2, 2, 1, 2)
print("group means:", t4_mu.reshape(-1).tolist())  # -> [2.5, 6.5]
print("group variances:", t4_var.reshape(-1).tolist())  # -> [1.25, 1.25]
print("group means after:", np.round(t4_means_after.reshape(-1), 6).tolist())  # -> [0.0, 0.0]
print("group variances after:", np.round(t4_vars_after.reshape(-1), 3).tolist())  # -> [1.0, 1.0]

assert np.allclose(t4_means_after, 0.0, atol=1e-6)
assert np.allclose(t4_vars_after, 1.0, atol=1e-5)

plt.figure(figsize=(4.6, 2.8))
plt.bar(["group 0", "group 1"], t4_mu.reshape(-1), color=["steelblue", "darkorange"])
plt.ylabel("raw mean")
plt.title("Toy 4 · separate statistics per group")
plt.show()

▶ What you'll see: channels 0-1 and channels 2-3 get their own means before being put back together.

### ✍️ Toy 5 · Gamma and beta restore scale and shift

After normalization, learnable `gamma` and `beta` can amplify, damp, shift, or zero out selected features.

In [ ]:
import numpy as np                              # arrays and elementwise affine maps for this toy.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_xhat = np.array([-1.0, 0.0, 1.0, -0.5, 0.5, 1.5])  # normalized coordinates.
t5_gamma = np.array([1.0, 2.0, 0.5, -1.0, 0.0, 1.0])  # learned scales.
t5_beta = np.array([0.0, 0.5, -0.5, 1.0, 2.0, -1.0])  # learned shifts.
t5_scaled = t5_gamma * t5_xhat                  # featurewise scaling               # -> [-1.0, 0.0, 0.5, 0.5, 0.0, 1.5]
t5_y = t5_scaled + t5_beta                      # affine output                     # -> [-1.0, 0.5, 0.0, 1.5, 2.0, 0.5]

print("x_hat:", t5_xhat.tolist())              # -> [-1.0, 0.0, 1.0, -0.5, 0.5, 1.5]
print("gamma:", t5_gamma.tolist())             # -> [1.0, 2.0, 0.5, -1.0, 0.0, 1.0]
print("beta:", t5_beta.tolist())               # -> [0.0, 0.5, -0.5, 1.0, 2.0, -1.0]
print("gamma*x_hat:", t5_scaled.tolist())      # -> [-1.0, 0.0, 0.5, 0.5, 0.0, 1.5]
print("gamma*x_hat+beta:", t5_y.tolist())      # -> [-1.0, 0.5, 0.0, 1.5, 2.0, 0.5]

assert np.allclose(t5_y, [-1.0, 0.5, 0.0, 1.5, 2.0, 0.5])

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t5_xhat.size) - 0.18, t5_xhat, width=0.36, color="gray", label="x_hat")
plt.bar(np.arange(t5_y.size) + 0.18, t5_y, width=0.36, color="seagreen", label="affine")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("feature")
plt.ylabel("value")
plt.title("Toy 5 · affine parameters restore freedom")
plt.legend()
plt.show()

▶ What you'll see: normalized coordinates can be reshaped into task-specific values after the affine step.

### ✍️ Toy 6 · Epsilon damps a nearly constant group

When variance is tiny, epsilon keeps microscopic differences from turning into huge standardized values.

In [ ]:
import numpy as np                              # arrays and numerical guards for this toy.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_flat = np.array([3.0, 3.0, 3.0, 3.001, 2.999, 3.0])  # nearly constant activations.
t6_mu = np.mean(t6_flat)                        # mean                              # -> 3.0
t6_var = np.var(t6_flat)                        # tiny variance                     # -> 0.0000003333
t6_without = (t6_flat - t6_mu) / np.sqrt(t6_var)          # no epsilon             # -> [0.0, 0.0, 0.0, 1.732, -1.732, 0.0]
t6_with = (t6_flat - t6_mu) / np.sqrt(t6_var + 1e-3)      # epsilon in denominator # -> [0.0, 0.0, 0.0, 0.032, -0.032, 0.0]

print("flat values:", t6_flat.tolist())        # -> [3.0, 3.0, 3.0, 3.001, 2.999, 3.0]
print("mean:", round(float(t6_mu), 6))          # -> 3.0
print("variance:", round(float(t6_var), 10))    # -> 0.0000003333
print("without epsilon:", np.round(t6_without, 3).tolist())  # -> [0.0, 0.0, 0.0, 1.732, -1.732, 0.0]
print("with epsilon:", np.round(t6_with, 3).tolist())        # -> [0.0, 0.0, 0.0, 0.032, -0.032, 0.0]

assert np.max(np.abs(t6_with)) < np.max(np.abs(t6_without))

plt.figure(figsize=(4.8, 2.8))
plt.plot(t6_without, marker="o", label="without eps", color="crimson")
plt.plot(t6_with, marker="s", label="with eps", color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("feature")
plt.ylabel("normalized value")
plt.title("Toy 6 · epsilon prevents a spike")
plt.legend()
plt.show()

▶ What you'll see: the same tiny deviations shrink from ±1.732 to about ±0.032 when epsilon is added.

### ✍️ Toy 7 · Per-example statistics ignore batch neighbors

Layer norm computes statistics inside each row, so changing another row in the minibatch does not change the first row's output.

In [ ]:
import numpy as np                              # arrays and per-row normalization for this toy.

t7_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t7_first = np.array([1., 2., 3., 4.])           # the example we track.
t7_neighbor_a = np.array([10., 12., 14., 16.])  # one possible batch neighbor.
t7_neighbor_b = np.array([-5., 0., 5., 10.])    # a very different batch neighbor.
t7_batch_a = np.stack([t7_first, t7_neighbor_a])  # batch A.
t7_batch_b = np.stack([t7_first, t7_neighbor_b])  # batch B.
t7_mu_a = np.mean(t7_batch_a, axis=1, keepdims=True)       # row means in batch A.
t7_var_a = np.var(t7_batch_a, axis=1, keepdims=True)       # row variances in batch A.
t7_mu_b = np.mean(t7_batch_b, axis=1, keepdims=True)       # row means in batch B.
t7_var_b = np.var(t7_batch_b, axis=1, keepdims=True)       # row variances in batch B.
t7_out_a = (t7_batch_a - t7_mu_a) / np.sqrt(t7_var_a + 1e-5)  # layer norm batch A.
t7_out_b = (t7_batch_b - t7_mu_b) / np.sqrt(t7_var_b + 1e-5)  # layer norm batch B.

print("batch A first row:", t7_batch_a[0].tolist())          # -> [1.0, 2.0, 3.0, 4.0]
print("batch B first row:", t7_batch_b[0].tolist())          # -> [1.0, 2.0, 3.0, 4.0]
print("first output in A:", np.round(t7_out_a[0], 3).tolist())  # -> [-1.342, -0.447, 0.447, 1.342]
print("first output in B:", np.round(t7_out_b[0], 3).tolist())  # -> [-1.342, -0.447, 0.447, 1.342]
print("outputs equal:", bool(np.allclose(t7_out_a[0], t7_out_b[0])))  # -> True

assert np.allclose(t7_out_a[0], t7_out_b[0])

plt.figure(figsize=(4.8, 2.8))
plt.plot(t7_out_a[0], marker="o", color="teal", label="first in batch A")
plt.plot(t7_out_b[0], marker="s", color="orange", linestyle="--", label="first in batch B")
plt.xlabel("feature")
plt.ylabel("layer-normalized value")
plt.title("Toy 7 · neighbor changes do not matter")
plt.legend()
plt.show()

▶ What you'll see: the two first-row curves sit exactly on top of each other.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, axis reductions, broadcasting, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib so every normalization statistic can be inspected visually.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def normalize_axes(x, axes, eps=1e-5): # normalize an array over chosen axes using NumPy broadcasting.
    mean = np.mean(x, axis=axes, keepdims=True) # compute the mean over exactly the requested axes.
    var = np.var(x, axis=axes, keepdims=True) # compute the variance over the same axes as the mean.
    return (x - mean) / np.sqrt(var + eps), mean, var # return normalized values plus the statistics for inspection.

def layer_norm_np(x, eps=1e-5): # layer norm for a 2-D batch shaped batch by features.
    return normalize_axes(x, axes=1, eps=eps)[0] # normalize each example across its feature axis.

def instance_norm_np(x, eps=1e-5): # instance norm for image-like tensors shaped N,C,H,W.
    return normalize_axes(x, axes=(2, 3), eps=eps)[0] # normalize each image-channel over spatial locations.

def group_norm_np(x, groups, eps=1e-5): # group norm for N,C,H,W tensors with C divisible by groups.
    n, c, h, w = x.shape # read tensor dimensions so the group axis can be created.
    grouped = x.reshape(n, groups, c // groups, h, w) # split channels into groups.
    y, _, _ = normalize_axes(grouped, axes=(2, 3, 4), eps=eps) # normalize each example-group.
    return y.reshape(x.shape) # return to the original N,C,H,W shape.

## 🟢 Basics (warm-up)

### Basic 1 — Center one vector

**Goal.** Subtract a mean from one activation vector, because normalization begins by removing a local offset. We build it in 2 steps.

In [ ]:
x_b1 = np.array([2.0, 4.0, 6.0, 8.0]) # create a simple feature vector with a positive offset.
mean_b1 = np.mean(x_b1) # compute the average feature value.
centered_b1 = x_b1 - mean_b1 # subtract the average from every coordinate.

print("mean:", mean_b1) # inspect the offset being removed.
print("centered:", centered_b1) # inspect the zero-centered vector.

assert mean_b1 == 5.0

▶ What you'll see: the vector is shifted from values around 5 to values around 0.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact bar chart.
plt.bar(["raw mean", "centered mean"], [np.mean(x_b1), np.mean(centered_b1)], color=["gray", "teal"]) # compare means before and after centering.
plt.title("Basic 1: centering removes offset") # title the plot.
plt.ylabel("mean") # label the mean axis.
plt.show() # display the chart.

▶ What you'll see: the centered mean bar lands exactly on zero.

👀 Takeaway: centering makes activations describe deviations from their local average.

### Basic 2 — Scale one vector by its standard deviation

**Goal.** Divide centered values by their standard deviation, because normalized units should mean comparable fluctuation sizes. We build it in 2 steps.

In [ ]:
x_b2 = np.array([2.0, 4.0, 6.0, 8.0]) # reuse a four-feature vector.
centered_b2 = x_b2 - np.mean(x_b2) # remove the feature mean first.
std_b2 = np.sqrt(np.var(x_b2) + 1e-5) # compute a stable standard deviation.
z_b2 = centered_b2 / std_b2 # convert raw deviations into standard-deviation units.

print("std:", round(float(std_b2), 3)) # inspect the scale divisor.
print("z:", np.round(z_b2, 3)) # inspect standardized values.

assert round(float(np.var(z_b2)), 3) == 1.0

▶ What you'll see: the standardized vector has variance about 1.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact visualization.
plt.plot(x_b2, marker="o", label="raw", color="gray") # plot raw values.
plt.plot(z_b2, marker="s", label="standardized", color="teal") # plot standardized values.
plt.axhline(0, color="black", linewidth=0.7) # show the centered baseline.
plt.title("Basic 2: values become z-like scores") # title the plot.
plt.legend() # show curve labels.
plt.show() # display the chart.

▶ What you'll see: the standardized curve keeps order but lives on a stable scale.

👀 Takeaway: division by standard deviation turns arbitrary activation units into comparable normalized units.

### Basic 3 — Add epsilon for numerical stability

**Goal.** See how epsilon protects nearly constant activations, because tiny variance can make normalized values explode. We build it in 2 steps.

In [ ]:
x_b3 = np.array([3.0, 3.0, 3.0, 3.001]) # create an almost-constant vector.
var_b3 = np.var(x_b3) # compute its tiny variance.
z_small_eps_b3 = (x_b3 - np.mean(x_b3)) / np.sqrt(var_b3 + 1e-5) # normalize with a small epsilon.
z_big_eps_b3 = (x_b3 - np.mean(x_b3)) / np.sqrt(var_b3 + 1e-2) # normalize with a larger epsilon.

print("variance:", round(float(var_b3), 8)) # inspect how small the variance is.
print("max |z| small eps:", round(float(np.max(np.abs(z_small_eps_b3))), 3)) # inspect the less-damped result.
print("max |z| big eps:", round(float(np.max(np.abs(z_big_eps_b3))), 3)) # inspect the more-damped result.

assert np.max(np.abs(z_big_eps_b3)) < np.max(np.abs(z_small_eps_b3))

▶ What you'll see: a larger epsilon makes the normalized values less extreme.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact comparison plot.
plt.bar(["eps=1e-5", "eps=1e-2"], [np.max(np.abs(z_small_eps_b3)), np.max(np.abs(z_big_eps_b3))], color=["orange", "teal"]) # compare normalized magnitudes.
plt.title("Basic 3: epsilon damps tiny-variance groups") # title the chart.
plt.ylabel("max absolute normalized value") # label the magnitude axis.
plt.show() # display the chart.

▶ What you'll see: the larger-epsilon bar is much shorter.

👀 Takeaway: epsilon is a stability constant that prevents division by an almost-zero standard deviation.

### Basic 4 — Normalize each row with layer norm

**Goal.** Normalize a batch across features per example, because layer norm should not mix examples together. We build it in 2 steps.

In [ ]:
X_b4 = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 12.0, 14.0, 16.0]]) # two examples with different scales.
Y_b4 = layer_norm_np(X_b4) # normalize each row independently across features.

print("row means:", np.round(np.mean(Y_b4, axis=1), 6)) # verify each row is centered.
print("row vars:", np.round(np.var(Y_b4, axis=1), 3)) # verify each row has unit variance.

assert np.allclose(np.mean(Y_b4, axis=1), 0.0, atol=1e-6)

▶ What you'll see: both rows have output mean 0 even though their raw means differ.

In [ ]:
plt.figure(figsize=(5, 3)) # create a row-wise comparison plot.
plt.plot(Y_b4[0], marker="o", label="example 0", color="teal") # plot normalized first row.
plt.plot(Y_b4[1], marker="s", label="example 1", color="orange") # plot normalized second row.
plt.title("Basic 4: layer norm outputs per row") # title the plot.
plt.xlabel("feature") # label features.
plt.ylabel("normalized value") # label normalized scale.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the normalized row shapes align because each row was standardized separately.

👀 Takeaway: layer norm computes statistics inside each example's feature vector.

### Basic 5 — Preserve feature order after normalization

**Goal.** Check that normalization changes scale without changing rank order inside a vector, because the relative pattern still matters. We build it in 2 steps.

In [ ]:
x_b5 = np.array([5.0, 1.0, 3.0, 7.0]) # create features with a clear ordering.
z_b5, _, _ = normalize_axes(x_b5, axes=0) # normalize across the whole vector.
order_raw_b5 = np.argsort(x_b5) # compute raw feature order.
order_norm_b5 = np.argsort(z_b5) # compute normalized feature order.

print("raw order:", order_raw_b5) # inspect raw rank order.
print("norm order:", order_norm_b5) # inspect normalized rank order.

assert np.array_equal(order_raw_b5, order_norm_b5)

▶ What you'll see: the feature ordering is unchanged by centering and positive scaling.

In [ ]:
plt.figure(figsize=(4, 3)) # create a rank-preservation plot.
plt.scatter(x_b5, z_b5, color="purple", s=80) # compare raw and normalized values coordinate by coordinate.
plt.title("Basic 5: normalization is monotone within one group") # title the scatter.
plt.xlabel("raw value") # label raw axis.
plt.ylabel("normalized value") # label normalized axis.
plt.show() # display the scatter.

▶ What you'll see: larger raw values correspond to larger normalized values.

👀 Takeaway: normalization preserves within-group ordering while changing offset and scale.

### Basic 6 — Use gamma and beta

**Goal.** Apply a learnable affine transformation after normalization, because networks need to recover task-specific scale and shift. We build it in 2 steps.

In [ ]:
z_b6 = np.array([-1.0, 0.0, 1.0]) # create three normalized coordinates.
gamma_b6 = np.array([2.0, 0.5, 1.0]) # define learned scales.
beta_b6 = np.array([0.0, 1.0, -1.0]) # define learned shifts.
y_b6 = gamma_b6 * z_b6 + beta_b6 # apply the affine restoration.

print("after affine:", y_b6) # inspect the rescaled and shifted values.

assert np.allclose(y_b6, [-2.0, 1.0, 0.0])

▶ What you'll see: different features receive different restored scales and offsets.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact affine comparison.
plt.bar(["z0", "z1", "z2"], z_b6, alpha=0.6, label="normalized", color="gray") # draw normalized values.
plt.plot(["z0", "z1", "z2"], y_b6, marker="o", label="affine", color="teal") # overlay affine outputs.
plt.axhline(0, color="black", linewidth=0.7) # show zero reference.
plt.title("Basic 6: gamma and beta reshape normalized values") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the affine output no longer has to keep the normalized scale.

👀 Takeaway: normalization stabilizes activations, while gamma and beta keep the representation flexible.

### Basic 7 — Normalize one image channel with instance norm

**Goal.** Normalize spatial pixels inside one image-channel, because instance norm treats each channel of each image separately. We build it in 2 steps.

In [ ]:
X_b7 = np.array([[[[1.0, 2.0], [3.0, 4.0]]]]) # shape N=1,C=1,H=2,W=2.
Y_b7 = instance_norm_np(X_b7) # normalize over H and W for the single channel.

print("spatial mean:", round(float(np.mean(Y_b7[0, 0])), 6)) # verify spatial centering.
print("spatial var:", round(float(np.var(Y_b7[0, 0])), 3)) # verify spatial unit variance.

assert round(float(np.var(Y_b7[0, 0])), 3) == 1.0

▶ What you'll see: the four pixels in the channel become mean 0 and variance 1.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.5)) # create raw and normalized heatmaps.
ax[0].imshow(X_b7[0, 0], cmap="viridis"); ax[0].set_title("raw") # show raw channel.
ax[1].imshow(Y_b7[0, 0], cmap="coolwarm"); ax[1].set_title("instance norm") # show normalized channel.
for a_b7 in ax:
    a_b7.set_xticks([]); a_b7.set_yticks([])
plt.show() # display the side-by-side heatmaps.

▶ What you'll see: the spatial pattern remains, but values are expressed as relative contrast.

👀 Takeaway: instance norm standardizes each image-channel's spatial pattern independently.

### Basic 8 — Split channels into groups

**Goal.** Reshape channels to expose group structure, because group norm computes statistics per group. We build it in 2 steps.

In [ ]:
X_b8 = np.arange(1, 17, dtype=float).reshape(1, 4, 2, 2) # create a 1x4x2x2 activation block.
groups_b8 = 2 # choose two channel groups.
Xg_b8 = X_b8.reshape(1, groups_b8, 2, 2, 2) # reshape channels into groups of two.

print("original shape:", X_b8.shape) # inspect the original tensor shape.
print("grouped shape:", Xg_b8.shape) # inspect the explicit group-axis shape.

assert Xg_b8.shape == (1, 2, 2, 2, 2)

▶ What you'll see: the channel axis is split into group and channels-per-group axes.

In [ ]:
group_means_b8 = np.mean(Xg_b8, axis=(2, 3, 4)) # compute one mean per group.

print("group means:", group_means_b8.reshape(-1)) # inspect the two group statistics.

plt.figure(figsize=(4, 3)) # create a compact bar chart.
plt.bar(["group 0", "group 1"], group_means_b8.reshape(-1), color=["steelblue", "orange"]) # plot group means.
plt.title("Basic 8: group statistics") # title the plot.
plt.ylabel("raw mean") # label the statistic.
plt.show() # display the chart.

▶ What you'll see: each channel group has its own mean before normalization.

👀 Takeaway: group norm starts by turning channels into explicit groups before reducing over them.

### Basic 9 — Compare group norm endpoints

**Goal.** Show that group norm connects layer-like and instance-like behavior, because the number of groups controls which axes are shared. We build it in 2 steps.

In [ ]:
X_b9 = np.arange(1, 17, dtype=float).reshape(1, 4, 2, 2) # create a 4-channel image block.
Y_one_group_b9 = group_norm_np(X_b9, groups=1) # one group uses all channels and pixels per example.
Y_four_groups_b9 = group_norm_np(X_b9, groups=4) # four groups use one channel per group, like instance norm.

print("one-group global mean:", round(float(np.mean(Y_one_group_b9)), 6)) # verify example-wide centering.
print("four-group channel means:", np.round(np.mean(Y_four_groups_b9, axis=(2, 3)).reshape(-1), 6)) # verify per-channel centering.

assert round(float(np.mean(Y_one_group_b9)), 6) == 0.0

▶ What you'll see: one group centers the whole example, while four groups center each channel.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact comparison.
plt.bar(["groups=1 var", "groups=4 ch0 var"], [np.var(Y_one_group_b9), np.var(Y_four_groups_b9[0, 0])], color=["teal", "purple"]) # compare normalized variances.
plt.title("Basic 9: group count changes the statistic") # title the plot.
plt.ylabel("variance") # label variance axis.
plt.show() # display the chart.

▶ What you'll see: both selected normalized regions have variance about 1, but the regions are different.

👀 Takeaway: group count is the knob that moves group norm between broad and narrow statistics.

### Basic 10 — Check batch independence

**Goal.** Verify that layer norm for one example does not change when another example changes, because statistics are computed per example. We build it in 2 steps.

In [ ]:
A_b10 = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 10.0, 10.0, 10.0]]) # first batch with one companion example.
B_b10 = np.array([[1.0, 2.0, 3.0, 4.0], [-5.0, 0.0, 5.0, 10.0]]) # second batch changes only the companion.
first_A_b10 = layer_norm_np(A_b10)[0] # normalize the shared first example in batch A.
first_B_b10 = layer_norm_np(B_b10)[0] # normalize the shared first example in batch B.

print("first A:", np.round(first_A_b10, 3)) # inspect first normalized result.
print("first B:", np.round(first_B_b10, 3)) # inspect second normalized result.

assert np.allclose(first_A_b10, first_B_b10)

▶ What you'll see: the shared first example has exactly the same normalized values in both batches.

In [ ]:
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.plot(first_A_b10, marker="o", label="batch A", color="teal") # plot normalized first example from batch A.
plt.plot(first_B_b10, marker="s", linestyle="--", label="batch B", color="orange") # plot normalized first example from batch B.
plt.title("Basic 10: layer norm is batch-independent") # title the chart.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: the two curves lie on top of each other.

👀 Takeaway: non-batch normalization variants can behave consistently even when minibatch composition changes.

## 🟡 Easy

### Easy 1 — Implement layer norm on a mini-batch

**Goal.** Write the layer-normalization calculation explicitly, because the row-wise axes define the algorithm. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1.0, 3.0, 5.0], [2.0, 2.0, 8.0], [10.0, 12.0, 14.0]]) # three examples by three features.
mean_e1 = np.mean(X_e1, axis=1, keepdims=True) # compute one mean per example.
var_e1 = np.var(X_e1, axis=1, keepdims=True) # compute one variance per example.

print("means:", mean_e1.ravel()) # inspect row statistics.
print("vars:", np.round(var_e1.ravel(), 3)) # inspect row variances.

▶ What you'll see: every row has its own statistic, not one statistic for the whole batch.

In [ ]:
Y_e1 = (X_e1 - mean_e1) / np.sqrt(var_e1 + 1e-5) # apply layer normalization with broadcasting.

print("output means:", np.round(np.mean(Y_e1, axis=1), 6)) # verify row centering.
print("output vars:", np.round(np.var(Y_e1, axis=1), 3)) # verify row scaling.

assert np.allclose(np.mean(Y_e1, axis=1), 0.0, atol=1e-6)

In [ ]:
plt.figure(figsize=(5, 3)) # create a heatmap of normalized rows.
plt.imshow(Y_e1, cmap="coolwarm", aspect="auto") # visualize standardized activations.
plt.colorbar(label="normalized value") # add color scale.
plt.title("Easy 1: layer-normalized mini-batch") # title the plot.
plt.xlabel("feature") # label feature axis.
plt.ylabel("example") # label example axis.
plt.show() # display the heatmap.

▶ What you'll see: each row contains negative and positive deviations around zero.

👀 Takeaway: layer norm is a row-wise standardization for feature vectors.

### Easy 2 — Implement instance norm on two channels

**Goal.** Normalize each image-channel across spatial positions, because instance norm removes per-channel image style. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[[[1.0, 2.0], [3.0, 4.0]], [[4.0, 4.0], [8.0, 8.0]]],
                 [[[10.0, 12.0], [14.0, 16.0]], [[1.0, 3.0], [5.0, 7.0]]]]) # shape 2,2,2,2.
mean_e2 = np.mean(X_e2, axis=(2, 3), keepdims=True) # one mean per image-channel.
var_e2 = np.var(X_e2, axis=(2, 3), keepdims=True) # one variance per image-channel.

print("stat shape:", mean_e2.shape) # inspect broadcasting shape.

▶ What you'll see: statistics have shape `(2, 2, 1, 1)`, one per image-channel.

In [ ]:
Y_e2 = (X_e2 - mean_e2) / np.sqrt(var_e2 + 1e-5) # apply instance normalization.
channel_means_e2 = np.mean(Y_e2, axis=(2, 3)) # compute output means per image-channel.
channel_vars_e2 = np.var(Y_e2, axis=(2, 3)) # compute output variances per image-channel.

print("channel means:\n", np.round(channel_means_e2, 6)) # inspect centering.
print("channel vars:\n", np.round(channel_vars_e2, 3)) # inspect scaling.

assert np.allclose(channel_means_e2, 0.0, atol=1e-6)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.5)) # create a before-after channel view.
ax[0].imshow(X_e2[1, 0], cmap="viridis"); ax[0].set_title("raw image1 ch0") # plot raw channel.
ax[1].imshow(Y_e2[1, 0], cmap="coolwarm"); ax[1].set_title("instance norm") # plot normalized channel.
for a_e2 in ax:
    a_e2.set_xticks([]); a_e2.set_yticks([])
plt.show() # display heatmaps.

▶ What you'll see: the channel's spatial pattern is preserved as relative high and low pixels.

👀 Takeaway: instance norm does not compare channels to each other; it normalizes each channel's spatial map.

### Easy 3 — Implement group norm by reshaping

**Goal.** Normalize channel groups explicitly, because group norm is easiest to understand after a reshape. We build it in 3 steps.

In [ ]:
X_e3 = np.arange(1, 33, dtype=float).reshape(1, 4, 2, 4) # create one example with four channels.
groups_e3 = 2 # split four channels into two groups.
N_e3, C_e3, H_e3, W_e3 = X_e3.shape # read dimensions for reshaping.
Xg_e3 = X_e3.reshape(N_e3, groups_e3, C_e3 // groups_e3, H_e3, W_e3) # expose groups.

print("grouped shape:", Xg_e3.shape) # inspect group shape.

▶ What you'll see: the tensor is reshaped into `(example, group, channels_per_group, height, width)`.

In [ ]:
mean_e3 = np.mean(Xg_e3, axis=(2, 3, 4), keepdims=True) # compute one mean per group.
var_e3 = np.var(Xg_e3, axis=(2, 3, 4), keepdims=True) # compute one variance per group.
Y_e3 = ((Xg_e3 - mean_e3) / np.sqrt(var_e3 + 1e-5)).reshape(X_e3.shape) # normalize and reshape back.
check_e3 = Y_e3.reshape(N_e3, groups_e3, C_e3 // groups_e3, H_e3, W_e3) # regroup output for checks.

print("output group means:", np.round(np.mean(check_e3, axis=(2, 3, 4)).reshape(-1), 6)) # verify group centering.

assert np.allclose(np.mean(check_e3, axis=(2, 3, 4)), 0.0, atol=1e-6)

In [ ]:
plt.figure(figsize=(5, 3)) # create a group-stat plot.
plt.bar(["group 0", "group 1"], mean_e3.reshape(-1), color=["steelblue", "orange"]) # show raw group means.
plt.title("Easy 3: raw means before group norm") # title the chart.
plt.ylabel("mean") # label the statistic.
plt.show() # display the chart.

▶ What you'll see: group statistics differ before normalization, then each group is centered in the check.

👀 Takeaway: group norm is just reshape, normalize within each group, and reshape back.

### Easy 4 — Compare LN, GN, and IN on one tensor

**Goal.** Apply all three variants to the same image tensor, because their outputs differ only through their statistic axes. We build it in 3 steps.

In [ ]:
X_e4 = np.arange(1, 17, dtype=float).reshape(1, 4, 2, 2) # one image-like activation tensor.
Y_ln_e4 = normalize_axes(X_e4, axes=(1, 2, 3))[0] # layer-style normalization over all channels and pixels for the example.
Y_gn_e4 = group_norm_np(X_e4, groups=2) # group normalization with two channel groups.
Y_in_e4 = instance_norm_np(X_e4) # instance normalization with one statistic per channel.

print("LN overall mean:", round(float(np.mean(Y_ln_e4)), 6)) # inspect layer-style centering.
print("IN channel means:", np.round(np.mean(Y_in_e4, axis=(2, 3)).reshape(-1), 6)) # inspect per-channel centering.

▶ What you'll see: LN centers the whole example; IN centers each channel.

In [ ]:
diff_gn_in_e4 = float(np.linalg.norm(Y_gn_e4 - Y_in_e4)) # measure how different group and instance outputs are.
diff_ln_gn_e4 = float(np.linalg.norm(Y_ln_e4 - Y_gn_e4)) # measure how different layer-style and group outputs are.

print("||GN-IN||:", round(diff_gn_in_e4, 3)) # inspect one axis-choice difference.
print("||LN-GN||:", round(diff_ln_gn_e4, 3)) # inspect another axis-choice difference.

assert diff_gn_in_e4 > 0.0 and diff_ln_gn_e4 > 0.0

In [ ]:
plt.figure(figsize=(5, 3)) # create a norm-difference bar chart.
plt.bar(["LN vs GN", "GN vs IN"], [diff_ln_gn_e4, diff_gn_in_e4], color=["teal", "purple"]) # compare output differences.
plt.title("Easy 4: same formula, different axes") # title the plot.
plt.ylabel("output L2 difference") # label difference scale.
plt.show() # display the chart.

▶ What you'll see: the same input produces distinct normalized outputs because the statistic regions differ.

👀 Takeaway: LN, GN, and IN are axis choices applied to the same standardization formula.

### Easy 5 — Visualize affine restoration

**Goal.** Add channel-wise gamma and beta after image normalization, because convolutional models usually learn one affine pair per channel. We build it in 3 steps.

In [ ]:
X_e5 = np.arange(1, 17, dtype=float).reshape(1, 4, 2, 2) # create one 4-channel activation block.
Y_e5 = group_norm_np(X_e5, groups=2) # normalize before affine restoration.
gamma_e5 = np.array([1.0, 0.5, 2.0, 0.0]).reshape(1, 4, 1, 1) # define channel-wise scales.
beta_e5 = np.array([0.0, 1.0, -1.0, 0.25]).reshape(1, 4, 1, 1) # define channel-wise shifts.

print("gamma shape:", gamma_e5.shape, "beta shape:", beta_e5.shape) # inspect broadcasting shape.

▶ What you'll see: gamma and beta are shaped to broadcast across height and width.

In [ ]:
Z_e5 = gamma_e5 * Y_e5 + beta_e5 # apply learned affine parameters channel-wise.
channel_means_e5 = np.mean(Z_e5, axis=(2, 3)).reshape(-1) # compute channel means after affine.

print("affine channel means:", np.round(channel_means_e5, 3)) # inspect restored shifts.

assert round(float(channel_means_e5[3]), 3) == 0.25

In [ ]:
plt.figure(figsize=(5, 3)) # create a channel-mean plot.
plt.bar(["ch0", "ch1", "ch2", "ch3"], channel_means_e5, color="seagreen") # show affine-shifted channel means.
plt.axhline(0, color="black", linewidth=0.7) # show zero reference.
plt.title("Easy 5: beta can restore nonzero channel means") # title the chart.
plt.ylabel("mean after affine") # label the statistic.
plt.show() # display the chart.

▶ What you'll see: normalized channels no longer have to stay mean-zero after beta is added.

👀 Takeaway: affine parameters let the model keep the optimization benefit without permanently fixing output scale.

## 🔴 Advanced

### Advanced 1 — Sweep group counts

**Goal.** Compare several group counts on the same tensor, because group count controls the size of the statistic pool. We build it in 4 steps.

In [ ]:
X_a1 = np.linspace(1.0, 64.0, 64).reshape(1, 8, 2, 4) # create one tensor with eight channels.
groups_grid_a1 = np.array([1, 2, 4, 8]) # test layer-like through instance-like group counts.
spread_a1 = [] # store output channel-mean spread for each group count.

print("group grid:", groups_grid_a1) # inspect the sweep values.

▶ What you'll see: the sweep will move from one broad group to one group per channel.

In [ ]:
for g_a1 in groups_grid_a1: # loop over candidate group counts.
    Y_a1 = group_norm_np(X_a1, groups=int(g_a1)) # apply group norm for this group count.
    ch_means_a1 = np.mean(Y_a1, axis=(2, 3)).reshape(-1) # summarize each output channel.
    spread_a1.append(float(np.std(ch_means_a1))) # measure how much channel means vary after normalization.

print("channel-mean spread:", np.round(spread_a1, 3)) # inspect axis-choice effect.

assert round(spread_a1[-1], 6) == 0.0

In [ ]:
plt.figure(figsize=(5, 3)) # create a sweep plot.
plt.plot(groups_grid_a1, spread_a1, marker="o", color="teal") # plot statistic-pool effect.
plt.title("Advanced 1: group count changes channel-mean spread") # title the plot.
plt.xlabel("number of groups") # label group-count axis.
plt.ylabel("std of channel means after GN") # label spread axis.
plt.xticks(groups_grid_a1) # show tested group counts.
plt.show() # display the plot.

▶ What you'll see: with one channel per group, every channel mean is forced to zero.

In [ ]:
samples_per_stat_a1 = X_a1.shape[1] // groups_grid_a1 * X_a1.shape[2] * X_a1.shape[3] # count values used per statistic.

print("values per mean/variance:", samples_per_stat_a1) # inspect sample-pool sizes.

assert samples_per_stat_a1[0] == 64 and samples_per_stat_a1[-1] == 8

▶ What you'll see: more groups means fewer values contribute to each mean and variance estimate.

👀 Takeaway: group count trades broad, stable statistics against channel-specific normalization.

### Advanced 2 — Show why batch statistics can be noisy

**Goal.** Contrast batch-dependent and layer-normalized outputs for one fixed example, because small batches can make batch statistics unstable. We build it in 4 steps.

In [ ]:
x_fixed_a2 = np.array([[1.0, 2.0, 3.0, 4.0]]) # the example whose output we track.
companions_a2 = [np.array([[10.0, 10.0, 10.0, 10.0]]), np.array([[-20.0, 0.0, 20.0, 40.0]])] # two different batch companions.

print("fixed example:", x_fixed_a2.ravel()) # inspect the unchanged example.

▶ What you'll see: the first example is identical in both upcoming batches.

In [ ]:
batch_outputs_a2 = [] # store batch-normalized first-example outputs.
layer_outputs_a2 = [] # store layer-normalized first-example outputs.
for comp_a2 in companions_a2: # compare two minibatch compositions.
    B_a2 = np.vstack([x_fixed_a2, comp_a2]) # create a two-example batch.
    batch_mu_a2 = np.mean(B_a2, axis=0, keepdims=True) # batch-style feature mean over examples.
    batch_var_a2 = np.var(B_a2, axis=0, keepdims=True) # batch-style feature variance over examples.
    batch_outputs_a2.append(((B_a2 - batch_mu_a2) / np.sqrt(batch_var_a2 + 1e-5))[0]) # save first example batch-normalized.
    layer_outputs_a2.append(layer_norm_np(B_a2)[0]) # save first example layer-normalized.

print("batch-dependent outputs:\n", np.round(batch_outputs_a2, 3)) # inspect changed outputs.
print("layer-norm outputs:\n", np.round(layer_outputs_a2, 3)) # inspect stable outputs.

assert np.allclose(layer_outputs_a2[0], layer_outputs_a2[1])

In [ ]:
change_batch_a2 = float(np.linalg.norm(batch_outputs_a2[0] - batch_outputs_a2[1])) # measure batch-normalized change.
change_layer_a2 = float(np.linalg.norm(layer_outputs_a2[0] - layer_outputs_a2[1])) # measure layer-normalized change.

print("batch-output change:", round(change_batch_a2, 3)) # inspect minibatch dependence.
print("layer-output change:", round(change_layer_a2, 3)) # inspect batch independence.

assert change_batch_a2 > change_layer_a2

In [ ]:
plt.figure(figsize=(5, 3)) # create a dependence comparison.
plt.bar(["batch-stat change", "layer-norm change"], [change_batch_a2, change_layer_a2], color=["crimson", "teal"]) # compare output changes.
plt.title("Advanced 2: layer norm avoids companion-example noise") # title the chart.
plt.ylabel("L2 change for fixed example") # label output-change scale.
plt.show() # display the chart.

▶ What you'll see: the batch-stat output changes when the companion changes, while layer norm stays fixed.

👀 Takeaway: per-example normalization is useful when minibatch statistics are unreliable or unavailable.

### Advanced 3 — Normalize a transformer-style sequence

**Goal.** Apply layer norm to a sequence tensor, because transformers normalize each token's hidden vector independently. We build it in 4 steps.

In [ ]:
X_a3 = np.array([[[1.0, 2.0, 3.0, 4.0], [10.0, 10.0, 12.0, 14.0], [0.5, 1.5, 2.5, 3.5]],
                 [[-1.0, 0.0, 1.0, 2.0], [4.0, 6.0, 8.0, 10.0], [3.0, 3.0, 3.0, 5.0]]]) # shape batch, tokens, hidden.
mean_a3 = np.mean(X_a3, axis=2, keepdims=True) # one mean per token vector.
var_a3 = np.var(X_a3, axis=2, keepdims=True) # one variance per token vector.

print("stat shape:", mean_a3.shape) # inspect token-wise statistic shape.

▶ What you'll see: statistics have shape `(batch, tokens, 1)`, one per token.

In [ ]:
Y_a3 = (X_a3 - mean_a3) / np.sqrt(var_a3 + 1e-5) # normalize each token hidden vector.
token_means_a3 = np.mean(Y_a3, axis=2) # compute output mean per token.
token_vars_a3 = np.var(Y_a3, axis=2) # compute output variance per token.

print("token means:\n", np.round(token_means_a3, 6)) # verify token centering.
print("token vars:\n", np.round(token_vars_a3, 3)) # verify token scaling.

assert np.allclose(token_means_a3, 0.0, atol=1e-6)

In [ ]:
gamma_a3 = np.array([1.0, 0.5, 2.0, 1.5]) # learned hidden-dimension scales.
beta_a3 = np.array([0.0, 0.1, -0.2, 0.3]) # learned hidden-dimension shifts.
Z_a3 = gamma_a3 * Y_a3 + beta_a3 # broadcast affine parameters across batch and tokens.

print("first token after affine:", np.round(Z_a3[0, 0], 3)) # inspect one token output.

assert Z_a3.shape == X_a3.shape

In [ ]:
plt.figure(figsize=(5, 3)) # create a token heatmap.
plt.imshow(Y_a3[0], cmap="coolwarm", aspect="auto") # visualize normalized hidden vectors for one sequence.
plt.colorbar(label="normalized value") # add color scale.
plt.title("Advanced 3: token-wise layer norm") # title the heatmap.
plt.xlabel("hidden dimension") # label hidden axis.
plt.ylabel("token") # label token axis.
plt.show() # display the heatmap.

▶ What you'll see: each token row has its own negative and positive hidden-dimension deviations.

👀 Takeaway: transformer layer norm standardizes each token vector without mixing tokens or batch examples.

### Advanced 4 — Track gradient-scale intuition with normalized inputs

**Goal.** Compare a linear gradient from raw and normalized activations, because input scale directly changes parameter update size. We build it in 4 steps.

In [ ]:
x_raw_a4 = np.array([100.0, 110.0, 120.0, 130.0]) # large-scale activations feeding one linear unit.
w_a4 = np.array([0.01, -0.02, 0.03, -0.01]) # small weights for a scalar prediction.
target_a4 = 1.0 # desired scalar output.
x_norm_a4, _, _ = normalize_axes(x_raw_a4, axes=0) # normalize the input vector.

print("raw norm:", round(float(np.linalg.norm(x_raw_a4)), 3)) # inspect raw vector length.
print("normalized norm:", round(float(np.linalg.norm(x_norm_a4)), 3)) # inspect normalized vector length.

▶ What you'll see: the normalized input has a much smaller, controlled vector norm.

In [ ]:
pred_raw_a4 = float(w_a4 @ x_raw_a4) # compute prediction from raw activations.
pred_norm_a4 = float(w_a4 @ x_norm_a4) # compute prediction from normalized activations.
err_raw_a4 = pred_raw_a4 - target_a4 # residual for raw-input squared loss.
err_norm_a4 = pred_norm_a4 - target_a4 # residual for normalized-input squared loss.
grad_raw_a4 = err_raw_a4 * x_raw_a4 # gradient with respect to weights for 0.5 error squared.
grad_norm_a4 = err_norm_a4 * x_norm_a4 # analogous gradient after normalization.

print("gradient norms:", round(float(np.linalg.norm(grad_raw_a4)), 3), round(float(np.linalg.norm(grad_norm_a4)), 3)) # inspect update scale.

assert np.linalg.norm(grad_norm_a4) < np.linalg.norm(grad_raw_a4)

In [ ]:
eta_a4 = 0.001 # choose a small learning rate.
step_raw_a4 = eta_a4 * grad_raw_a4 # raw-input weight step magnitude.
step_norm_a4 = eta_a4 * grad_norm_a4 # normalized-input weight step magnitude.

print("step norms:", round(float(np.linalg.norm(step_raw_a4)), 4), round(float(np.linalg.norm(step_norm_a4)), 4)) # compare update sizes.

In [ ]:
plt.figure(figsize=(5, 3)) # create a gradient-scale plot.
plt.bar(["raw input grad", "normalized input grad"], [np.linalg.norm(grad_raw_a4), np.linalg.norm(grad_norm_a4)], color=["crimson", "teal"]) # compare gradient norms.
plt.title("Advanced 4: normalization controls update scale") # title the chart.
plt.ylabel("gradient norm") # label gradient scale.
plt.show() # display the chart.

▶ What you'll see: the raw activation gradient is far larger because the input vector is far larger.

👀 Takeaway: normalization can make optimizer steps more predictable by controlling activation scale.

### Advanced 5 — Estimate activation memory and statistic cost

**Goal.** Count activation memory and statistic sizes, because normalization choices also matter for hardware and bookkeeping. We build it in 4 steps.

In [ ]:
N_a5, C_a5, H_a5, W_a5 = 8, 32, 16, 16 # define a small image activation block.
bytes_per_float_a5 = 4 # assume 32-bit floating point activations.
activation_kb_a5 = N_a5 * C_a5 * H_a5 * W_a5 * bytes_per_float_a5 / 1024 # compute activation memory in KB.

print("activation memory KB:", round(activation_kb_a5, 1)) # inspect storage for the tensor.

assert round(activation_kb_a5, 1) == 256.0

▶ What you'll see: even a modest activation block already uses 256 KB in float32.

In [ ]:
groups_a5 = 8 # choose group normalization with eight groups.
ln_stats_a5 = N_a5 # one mean and variance pair per example for layer-style over C,H,W.
gn_stats_a5 = N_a5 * groups_a5 # one pair per example-group.
in_stats_a5 = N_a5 * C_a5 # one pair per image-channel.

print("stat counts LN/GN/IN:", ln_stats_a5, gn_stats_a5, in_stats_a5) # inspect bookkeeping counts.

assert gn_stats_a5 == 64 and in_stats_a5 == 256

In [ ]:
stat_pairs_a5 = np.array([ln_stats_a5, gn_stats_a5, in_stats_a5]) # collect statistic counts.
stat_memory_kb_a5 = stat_pairs_a5 * 2 * bytes_per_float_a5 / 1024 # each statistic stores mean and variance.

print("stat memory KB:", np.round(stat_memory_kb_a5, 3)) # inspect mean/variance storage.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bookkeeping plot.
plt.bar(["LN", "GN", "IN"], stat_pairs_a5, color=["teal", "orange", "purple"]) # compare number of statistic regions.
plt.title("Advanced 5: statistic regions by normalization type") # title the chart.
plt.ylabel("mean/variance regions") # label count axis.
plt.show() # display the chart.

▶ What you'll see: instance norm tracks many more statistic regions than layer norm, with group norm in between.

👀 Takeaway: axis choices affect not only math behavior but also how many statistics must be computed and stored.